In [ ]:
from batched_drafter import BatchedDrafter
from transformers import AutoModelForCausalLM, AutoTokenizer
import drafter_pipeline as dp

In [ ]:
# Tokenizer
# left pad- so that all sequences end at same position - batched generation
tokenizer = AutoTokenizer.from_pretrained(
    model_name, use_fast = True, padding_side = 'left'
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id



In [ ]:
batched_draft = BatchedDrafter()
is_mps = batched_draft.is_mps
is_cuda = batched_draft.is_cuda
use_bnb_nf4 = batched_draft.use_bnb_nf4
use_int8 = dp.INT8_Q
DEVICE = dp.DEVICE

In [ ]:
def load_vllm(model_name):
    '''
    Load model via vLLM for continous batching
    -All m prompts share one scheduling window
    -Paged Attention
    -tensor parallel size = 1 (1 GPU)
    
    '''
    from vllm import LLM, SamplingParams
    vllm_llm = LLM(
        model = model_name,
        dtype = 'bfloat16',
        tensor_parallel_size = 1,
        max_model_len = dp.MAX_NEW_TOKENS + dp.MAX_INPUT_LEN, 
    )
    vllm_sampling = SamplingParams(
        temperature = dp.TEMPERATURE,
        max_tokens = dp.MAX_NEW_TOKENS,
        logprobs = 1, # collect top-1 logprob
    )
    logger.info('vLLM engine loaded: %s', model_name)

    
    

    